# 5-1. 어텐션 — 고정 문맥에서의 탈출

> **이 노트북에서 답할 질문**
> NPLM의 한계(고정 문맥, 위치별 가중치)를 어떤 장치가 어떻게 푸는가?

## 1. 다시 문제 정의

4-1 마지막에서 본 두 가지 한계입니다.

1. **문맥 길이가 고정** — `nn.Linear(ctx*d, hidden)` 이 박혀 있어서 길이를 못 바꿈
2. **위치마다 다른 가중치** — 같은 단어가 자리에 따라 다른 파라미터로 처리됨

원하는 것을 정확히 적어 봅시다.

> 길이가 몇이든 받아서, **중요한 단어를 골라 섞어 주는** 장치.
> 그 "고르는 방식"이 위치가 아니라 **내용**에 따라 결정되어야 한다.

## 2. 첫 걸음: 그냥 평균 내기

가장 단순한 "길이에 무관한" 방법은 **앞의 모든 단어 벡터를 평균**내는 것입니다.

$$h_t = \frac{1}{t}\sum_{i \le t} x_i$$

길이가 몇이든 작동하고, 파라미터도 필요 없습니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

In [ ]:
B, T, C = 1, 6, 4          # 배치, 시퀀스 길이, 채널(임베딩 차원)
x = torch.randn(B, T, C)

# 방법 A: 직관적인 반복문
out_loop = torch.zeros(B, T, C)
for t in range(T):
    out_loop[:, t] = x[:, :t+1].mean(dim=1)     # 0..t 까지 평균

# 방법 B: 행렬 곱 한 번으로
w = torch.tril(torch.ones(T, T))                # 아래 삼각 행렬
w = w / w.sum(dim=1, keepdim=True)              # 각 행의 합이 1이 되도록
out_mat = w @ x

print("가중치 행렬 w:")
print(w.numpy().round(3))
print(f"{chr(10)}두 방법이 같은가? {torch.allclose(out_loop, out_mat, atol=1e-6)}")

### 이 행렬이 전부입니다

`w` 를 보세요. **t번째 행은 "t번째 출력을 만들 때 각 입력을 얼마나 섞을지"** 입니다.

- 아래 삼각형 → **미래를 보지 않음** (다음 단어를 예측하는데 정답을 미리 보면 반칙)
- 각 행의 합이 1 → 가중 평균

문제는 이 가중치가 **전부 똑같다**는 것. `언어 모델은 다음에 올`에서
`모델은`과 `올`을 똑같은 비중으로 섞는 건 말이 안 되죠.

> **핵심 전환**: 이 가중치를 **데이터에서 계산**하게 만들자.
> 그것이 어텐션입니다.

## 3. Query, Key, Value

어떻게 "누구를 얼마나 볼지"를 계산할까요? 검색 엔진에 비유하면 이해가 빠릅니다.

| 이름 | 역할 | 비유 |
|---|---|---|
| **Query** (질의) | 내가 찾는 것 | 검색창에 친 검색어 |
| **Key** (키) | 내가 가진 것의 색인 | 문서의 제목/태그 |
| **Value** (값) | 실제로 전달할 내용 | 문서의 본문 |

각 토큰이 세 가지를 모두 내놓습니다. 그리고

1. 내 **Query**와 남들의 **Key**를 맞춰 본다 → 점수
2. 점수를 softmax로 확률화 → **어텐션 가중치**
3. 그 가중치로 남들의 **Value**를 가중 평균 → 출력

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
set_seed(42)
B, T, C = 1, 6, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
v = value(x)    # (B, T, 16)

# 1) Q와 K를 맞춰 점수 계산
scores = q @ k.transpose(-2, -1)          # (B, T, T)
print(f"scores.shape = {tuple(scores.shape)}   <- 모든 토큰 쌍의 점수")
print(f"scores[0,2,4] = 3번째 토큰이 5번째 토큰을 얼마나 볼지 = {scores[0,2,4].item():.3f}")

### 스케일링 — $\sqrt{d_k}$ 로 나누는 이유

$q \cdot k$ 는 `head_size` 개 항의 합입니다. 차원이 커지면 이 값의 **분산이 차원에 비례해 커집니다.**
값이 크면 softmax가 극단적으로 뾰족해져서, **한 곳만 보고 나머지는 0**이 됩니다.
그러면 기울기가 거의 흐르지 않아 학습이 안 됩니다.

직접 확인해 봅시다.

In [ ]:
for hs in (4, 16, 64, 256):
    qq = torch.randn(1000, hs)
    kk = torch.randn(1000, hs)
    raw = (qq * kk).sum(-1)
    print(f"head_size={hs:>4}:  점수 분산 {raw.var().item():7.2f}  ->  1/sqrt(d) 로 나누면 {(raw/math.sqrt(hs)).var().item():.2f}")

print(f"{chr(10)}softmax가 얼마나 뾰족해지는지:")
demo = torch.tensor([1.0, 0.5, 0.2, 0.0])
for scale in (1, 5, 20):
    p = F.softmax(demo * scale, dim=-1)
    print(f"  점수에 x{scale:>2}: {p.numpy().round(3)}   최댓값 {p.max():.3f}")
print(f"{chr(10)}-> 점수가 크면 사실상 argmax가 되어 버립니다. 스케일링이 이걸 막습니다.")

## 4. 인과 마스크(causal mask)

언어 모델은 **다음 단어를 예측**합니다. 그런데 어텐션은 기본적으로 모든 토큰을 봅니다.
미래를 보게 두면 정답을 컨닝하는 셈이죠.

그래서 미래 위치의 점수를 $-\infty$ 로 만듭니다. softmax를 지나면 정확히 0이 됩니다.

In [ ]:
scores = scores / math.sqrt(head_size)

tril = torch.tril(torch.ones(T, T))
masked = scores.masked_fill(tril == 0, float("-inf"))

print("마스크 적용 전 (앞 4x4):")
print(scores[0, :4, :4].detach().numpy().round(2))
print(f"{chr(10)}마스크 적용 후:")
print(masked[0, :4, :4].detach().numpy().round(2))

attn = F.softmax(masked, dim=-1)
print(f"{chr(10)}softmax 후 (= 어텐션 가중치):")
print(attn[0, :4, :4].detach().numpy().round(3))
print(f"{chr(10)}각 행의 합: {attn[0].sum(-1).detach().numpy().round(3)}   <- 전부 1")

In [ ]:
out = attn @ v
print(f"출력 shape: {tuple(out.shape)}   (입력 T={T} 과 같은 길이)")
print(f"{chr(10)}주목: 파라미터는 key/query/value 3개의 Linear 뿐입니다.")
print(f"      시퀀스 길이 T 는 파라미터에 전혀 나타나지 않습니다!")
print(f"      -> 학습 때 T=6 이었어도 추론 때 T=100 으로 쓸 수 있습니다.")

### **이것이 NPLM 문제의 해결입니다**

`nn.Linear(C, head_size)` — 어디에도 `T`가 없습니다.
**같은 가중치가 모든 위치에 공유**되고, 길이는 런타임에 정해집니다.

두 한계가 동시에 풀렸습니다:
- ✅ 문맥 길이 가변
- ✅ 위치가 아니라 **내용**으로 가중치 결정

## 5. 클래스로 정리

In [ ]:
class Head(nn.Module):
    """단일 어텐션 헤드 (인과 마스크 포함)."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # 학습 대상이 아닌 상수는 buffer 로 등록 (state_dict 에 함께 저장/이동됨)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5      # (B,T,T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        return att @ v, att

head = Head(n_embd=32, head_size=16, block_size=8)
xx = torch.randn(2, 5, 32)
out, att = head(xx)
print(f"입력 {tuple(xx.shape)} -> 출력 {tuple(out.shape)},  어텐션맵 {tuple(att.shape)}")

# 길이를 바꿔도 같은 모듈이 작동합니다
for T_try in (1, 3, 8):
    o, _ = head(torch.randn(1, T_try, 32))
    print(f"  T={T_try} -> 출력 {tuple(o.shape)}  (같은 파라미터로!)")

## 6. 어텐션 맵 들여다보기

어텐션의 좋은 점 하나는 **가중치를 눈으로 볼 수 있다**는 것입니다.
"이 토큰이 무엇을 보고 있는가"가 그대로 그림이 됩니다.

학습되지 않은 무작위 헤드와, **일부러 "직전 토큰을 보게" 만든 헤드**를 비교해 봅시다.

In [ ]:
import matplotlib.pyplot as plt

T_vis = 8
set_seed(0)
h = Head(n_embd=32, head_size=16, block_size=T_vis)
xv = torch.randn(1, T_vis, 32)
_, att_random = h(xv)

# 비교용: 직전 토큰만 보는 이상적 패턴 (학습이 잘 되면 이런 구조가 나타납니다)
ideal = torch.zeros(T_vis, T_vis)
for i in range(T_vis):
    ideal[i, max(i - 1, 0)] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, m, title in zip(axes,
                        [att_random[0].detach(), ideal],
                        ["학습 전 (무작위)", "직전 토큰 주목 패턴"]):
    im = ax.imshow(m, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel("보는 대상 (key)"); ax.set_ylabel("보는 주체 (query)")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print("왼쪽: 아직 아무 구조도 없지만 오른쪽 위 삼각형은 이미 0입니다 -- 인과 마스크가 작동 중.")
print("오른쪽: 실제 학습된 모델에서 흔히 관찰되는 패턴 중 하나입니다.")

## 7. 대가: $O(T^2)$

어텐션은 공짜가 아닙니다. **모든 토큰 쌍**의 점수를 계산하므로 $T \times T$ 행렬이 필요합니다.

In [ ]:
print(f"{'문맥 길이 T':>12} | {'어텐션 행렬 원소 수':>20} | {'fp32 메모리(헤드 1개)':>22}")
print("-" * 62)
for T_ex in (128, 1024, 8192, 131072):
    cells = T_ex * T_ex
    mb = cells * 4 / 1e6
    unit = f"{mb/1000:>16.1f} GB" if mb >= 1000 else f"{mb:>16.1f} MB"
    print(f"{T_ex:>12,} | {cells:>20,} | {unit:>22}")

print(f"{chr(10)}헤드가 여러 개, 층이 여러 개라는 걸 생각하면 이 값에 수십 배가 곱해집니다.")
print("'긴 문맥'이 비싼 기능인 이유이고, FlashAttention 같은 기법이 나온 이유입니다.")

### 그래서 나온 것들 (이름만 알아 두세요)

- **FlashAttention**: $T \times T$ 행렬을 통째로 메모리에 올리지 않고 블록 단위로 계산
- **Sliding window / Sparse attention**: 가까운 토큰만 보게 제한
- **GQA / MQA**: Key·Value를 헤드끼리 공유해 캐시를 줄임 (5-3에서 다룹니다)

파이토치는 이 최적화들을 `F.scaled_dot_product_attention` 에 담아 두었습니다.
**실무에서는 직접 짜지 말고 이걸 쓰세요.**

In [ ]:
# 우리 구현 vs 파이토치 내장 함수
set_seed(0)
B, T, hs = 2, 6, 16
q_ = torch.randn(B, T, hs); k_ = torch.randn(B, T, hs); v_ = torch.randn(B, T, hs)

# 직접 구현
att_ = (q_ @ k_.transpose(-2, -1)) * hs ** -0.5
att_ = att_.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))
mine = F.softmax(att_, dim=-1) @ v_

# 파이토치 내장 (is_causal=True 가 인과 마스크를 대신해 줍니다)
builtin = F.scaled_dot_product_attention(q_, k_, v_, is_causal=True)

print(f"결과가 같은가? {torch.allclose(mine, builtin, atol=1e-5)}")
print(f"최대 오차: {(mine - builtin).abs().max().item():.2e}")

---

## 정리

| | NPLM | 어텐션 |
|---|---|---|
| 문맥 길이 | 고정 | **가변** |
| 가중치 결정 | 위치별 고정 | **내용에 따라 동적** |
| 파라미터와 T | T에 비례 | **T와 무관** |
| 계산량 | $O(T)$ | $O(T^2)$ |

**어텐션 한 줄 요약**

$$\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V$$

- $QK^\top$: 누가 누구를 볼지 점수
- $\sqrt{d_k}$: softmax가 뾰족해지지 않게
- mask: 미래 차단
- $V$: 실제로 섞을 내용

## 연습

1. `Head` 에서 마스크를 빼면(양방향 어텐션) 언어 모델 학습이 왜 망가지는지 설명해 보세요.
   (힌트: BERT는 마스크가 없습니다. 대신 학습 방식이 다릅니다.)
2. `head_size` 를 4와 128로 바꿔 가며 어텐션 맵이 얼마나 뾰족해지는지 비교하세요.
3. 스케일링($\sqrt{d_k}$)을 빼고 학습을 돌리면 어떤 일이 생기는지 5-2 이후에 실험해 보세요.

## 다음

→ `0502_TransformerBlock.ipynb` : 헤드 여러 개 + 잔차 + 정규화 = 트랜스포머 블록